In [5]:
import pandas as pd

data = {
    "customer_id": [1, 2, 3, 4, 5],
    "first_name": ["Thomas", "Alice", "Timothy", "Bob", "Tracy"],
    "last_name": ["Taylor", "Smith", "Turner", "Thomas", "Jones"],
}
customer_df = pd.DataFrame(data)

mask = customer_df["last_name"].str.startswith("T").fillna(False)
filtered_df = customer_df[mask]

result = filtered_df[
    ["customer_id", "first_name", "last_name"]
].sort_values(by="first_name", ascending=True)

print(result)




   customer_id first_name last_name
3            4        Bob    Thomas
0            1     Thomas    Taylor
2            3    Timothy    Turner


In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

cursor.execute("CREATE TABLE film (film_id INT, title TEXT)")
cursor.execute("CREATE TABLE inventory (inventory_id INT, film_id INT)")
cursor.execute("CREATE TABLE rental (rental_id INT, inventory_id INT)")

cursor.executemany(
    "INSERT INTO film VALUES (?, ?)",
    [(1, "ACADEMY DINOSAUR"), (2, "ACE GOLDFINGER")],
)
cursor.executemany("INSERT INTO inventory VALUES (?, ?)", [(1, 1), (2, 2)])
cursor.executemany(
    "INSERT INTO rental VALUES (?, ?)", [(101, 1), (102, 1), (103, 2)]
)
conn.commit()

query = """
SELECT f.title, COUNT(r.rental_id) AS rental_count
FROM film f
JOIN inventory i ON f.film_id = i.film_id
JOIN rental r ON i.inventory_id = r.inventory_id
GROUP BY f.film_id, f.title
ORDER BY rental_count DESC
LIMIT 10;
"""

result_2 = pd.read_sql_query(query, conn)
conn.close()

print(result_2)



              title  rental_count
0  ACADEMY DINOSAUR             2
1    ACE GOLDFINGER             1


In [13]:
import pandas as pd


def get_top_rented_movies(rental_df, inventory_df, film_df):
    """Executes the exact relational joins and counts mapped above."""

    merged_data = rental_df.merge(inventory_df, on="inventory_id").merge(
        film_df, on="film_id"
    )

    result_2 = (
        merged_data.groupby(["film_id", "title"])["rental_id"]
        .count()
        .reset_index()
    )

    result_2 = result_2.rename(columns={"rental_id": "rental_count"})

    return result_2.sort_values(by="rental_count", ascending=False).head(10)


mock_rental = pd.DataFrame(
    {"rental_id": [101, 102, 103, 104], "inventory_id": [1, 1, 1, 2]}
)
mock_inventory = pd.DataFrame(
    {"inventory_id": [1, 2, 3], "film_id": [500, 600, 700]}
)
mock_film = pd.DataFrame(
    {
        "film_id": [500, 600, 700],
        "title": ["Inception", "Interstellar", "Memento"],
    }
)

top_movies = get_top_rented_movies(mock_rental, mock_inventory, mock_film)
print(top_movies)




   film_id         title  rental_count
0      500     Inception             3
1      600  Interstellar             1


In [16]:
import pandas as pd


def get_customer_spending(customer_df, payment_df):
    """Calculates total amount spent by each customer, ordered least to most."""

    merged_data = customer_df.merge(payment_df, on="customer_id")

    result = (
        merged_data.groupby(["customer_id", "first_name", "last_name"])[
            "amount"
        ]
        .sum()
        .reset_index()
    )

    result = result.rename(columns={"amount": "total_spent"})

    return result.sort_values(by="total_spent", ascending=True)

mock_customer = pd.DataFrame(
    {
        "customer_id": [1, 2],
        "first_name": ["Alice", "Bob"],
        "last_name": ["Smith", "Jones"],
    }
)
mock_payment = pd.DataFrame(
    {"customer_id": [1, 1, 2], "amount": [10.99, 5.00, 25.50]}
)

print("--- Customer Spending Results ---")
print(get_customer_spending(mock_customer, mock_payment))



--- Customer Spending Results ---
   customer_id first_name last_name  total_spent
0            1      Alice     Smith        15.99
1            2        Bob     Jones        25.50


In [17]:
import pandas as pd


def get_top_actor_2006(actor_df, film_actor_df, film_df):
    """Identifies the actor with the most movies released in the year 2006."""

    films_2006 = film_df[film_df["release_year"] == 2006]

    actor_map = actor_df.merge(film_actor_df, on="actor_id").merge(
        films_2006, on="film_id"
    )

    actor_map["actor_name"] = (
        actor_map["first_name"] + " " + actor_map["last_name"]
    )

    result = (
        actor_map.groupby(["actor_id", "actor_name"])["film_id"]
        .count()
        .reset_index()
    )
    result = result.rename(columns={"film_id": "total_movies"})

    return result.sort_values(by="total_movies", ascending=False).head(1)

mock_actor = pd.DataFrame(
    {"actor_id": [1, 2], "first_name": ["John", "Jane"], "last_name": ["Doe", "Smith"]}
)
mock_film_actor = pd.DataFrame(
    {"actor_id": [1, 1, 2], "film_id": [101, 102, 101]}
)
mock_film = pd.DataFrame(
    {"film_id": [101, 102], "release_year": [2006, 2006]}
)

print("\n--- Top Actor 2006 Results ---")
print(get_top_actor_2006(mock_actor, mock_film_actor, mock_film))




--- Top Actor 2006 Results ---
   actor_id actor_name  total_movies
0         1   John Doe             2


In [22]:
import sqlite3


def print_execution_plans_safe():
    """Builds a mock in-memory database to display the query execution plans."""

    conn = sqlite3.connect(":memory:")
    cursor = conn.cursor()

    cursor.execute(
        "CREATE TABLE customer (customer_id INT, first_name TEXT, last_name TEXT)"
    )
    cursor.execute("CREATE TABLE payment (customer_id INT, amount REAL)")
    cursor.execute("CREATE TABLE actor (actor_id INT, first_name TEXT, last_name TEXT)")
    cursor.execute("CREATE TABLE film_actor (actor_id INT, film_id INT)")
    cursor.execute("CREATE TABLE film (film_id INT, release_year INT)")

    # 3. Plan for Query #4
    print("--- Plan for Query #4 ---")
    cursor.execute(
        """
        EXPLAIN QUERY PLAN
        SELECT c.customer_id, c.first_name, c.last_name, SUM(p.amount) AS total_spent
        FROM customer c
        JOIN payment p ON c.customer_id = p.customer_id
        GROUP BY c.customer_id, c.first_name, c.last_name
        ORDER BY total_spent ASC;
    """
    )
    for row in cursor.fetchall():
        print(row)

    # 4. Plan for Query #5
    print("\n--- Plan for Query #5 ---")
    cursor.execute(
        """
        EXPLAIN QUERY PLAN
        SELECT a.first_name || ' ' || a.last_name AS actor_name, COUNT(fa.film_id) AS total_movies
        FROM actor a
        JOIN film_actor fa ON a.actor_id = fa.actor_id
        JOIN film f ON fa.film_id = f.film_id
        WHERE f.release_year = 2006
        GROUP BY a.actor_id, a.first_name, a.last_name
        ORDER BY total_movies DESC
        LIMIT 1;
    """
    )
    for row in cursor.fetchall():
        print(row)

    conn.close()

print_execution_plans_safe()



--- Plan for Query #4 ---
(8, 0, 216, 'SCAN c')
(12, 0, 0, 'BLOOM FILTER ON p (customer_id=?)')
(23, 0, 53, 'SEARCH p USING AUTOMATIC COVERING INDEX (customer_id=?)')
(30, 0, 0, 'USE TEMP B-TREE FOR GROUP BY')
(76, 0, 0, 'USE TEMP B-TREE FOR ORDER BY')

--- Plan for Query #5 ---
(10, 0, 216, 'SCAN f')
(16, 0, 0, 'BLOOM FILTER ON fa (film_id=?)')
(26, 0, 53, 'SEARCH fa USING AUTOMATIC COVERING INDEX (film_id=?)')
(35, 0, 0, 'BLOOM FILTER ON a (actor_id=?)')
(46, 0, 53, 'SEARCH a USING AUTOMATIC COVERING INDEX (actor_id=?)')
(53, 0, 0, 'USE TEMP B-TREE FOR GROUP BY')
(103, 0, 0, 'USE TEMP B-TREE FOR ORDER BY')


In [28]:
import pandas as pd


def get_avg_rental_rate_by_genre(category_df, film_category_df, film_df):
    """Calculates the average rental rate for each movie genre."""

    merged_data = category_df.merge(film_category_df, on="category_id").merge(
        film_df, on="film_id"
    )

    result = (
        merged_data.groupby("name")["rental_rate"].mean().reset_index()
    )

    result = result.rename(
        columns={"name": "genre", "rental_rate": "avg_rental_rate"}
    )
    result["avg_rental_rate"] = result["avg_rental_rate"].round(2)

    return result.sort_values(by="avg_rental_rate", ascending=False)

mock_category = pd.DataFrame(
    {"category_id": [1, 2], "name": ["Action", "Comedy"]}
)
mock_film_category = pd.DataFrame({"category_id": [1, 2, 1], "film_id": [10, 20, 30]})
mock_film = pd.DataFrame(
    {"film_id": [10, 20, 30], "rental_rate": [2.99, 4.99, 0.99]}
)

print("--- Average Rental Rate by Genre ---")
print(
    get_avg_rental_rate_by_genre(
        mock_category, mock_film_category, mock_film
    )
)




--- Average Rental Rate by Genre ---
    genre  avg_rental_rate
1  Comedy             4.99
0  Action             1.99


In [29]:
import pandas as pd


def get_top_5_categories(
    category_df, film_category_df, inventory_df, rental_df, payment_df
):
    """Finds top 5 categories with highest volume of rentals and total sales."""

    full_merge = (
        category_df.merge(film_category_df, on="category_id")
        .merge(inventory_df, on="film_id")
        .merge(rental_df, on="inventory_id")
        .merge(payment_df, on="rental_id")
    )

    result = (
        full_merge.groupby("name")
        .agg(total_rentals=("rental_id", "count"), total_sales=("amount", "sum"))
        .reset_index()
    )

    result = result.rename(columns={"name": "category_name"})

    return result.sort_values(by="total_rentals", ascending=False).head(5)

mock_cat = pd.DataFrame({"category_id": [1, 2], "name": ["Sci-Fi", "Drama"]})
mock_fc = pd.DataFrame({"category_id": [1, 2], "film_id": [101, 102]})
mock_inv = pd.DataFrame({"film_id": [101, 102], "inventory_id": [1001, 1002]})
mock_rent = pd.DataFrame({"inventory_id": [1001, 1002], "rental_id": [5001, 5002]})
mock_pay = pd.DataFrame({"rental_id": [5001, 5002], "amount": [4.99, 2.99]})

print("\n--- Top 5 Categories by Rentals and Sales ---")
print(
    get_top_5_categories(
        mock_cat, mock_fc, mock_inv, mock_rent, mock_pay
    )
)




--- Top 5 Categories by Rentals and Sales ---
  category_name  total_rentals  total_sales
0         Drama              1         2.99
1        Sci-Fi              1         4.99
